# video-descriptor

Sube un video, extrae el audio con `ffmpeg` y transcribe con WhisperX. Salidas: `JSON`, `TXT`, `SRT` y `VTT`.

Antes de ejecutar: `Runtime > Change runtime type > GPU`.

In [ ]:
import os
import subprocess
import sys
import time

MARKER = '/content/.video_descriptor_install_done'
REQ_URL = 'https://raw.githubusercontent.com/pballestawork/video-descriptor/main/requirements-colab.txt'
REPO_URL = 'git+https://github.com/pballestawork/video-descriptor.git'

def run_step(name, command, *, shell=False, check=True):
    print(f'\n=== {name} ===', flush=True)
    print(command if isinstance(command, str) else ' '.join(command), flush=True)
    start = time.monotonic()
    result = subprocess.run(command, shell=shell, check=check)
    print(f'=== {name} finished in {time.monotonic() - start:.1f}s ===', flush=True)
    return result

if not os.path.exists(MARKER):
    run_step('apt update', 'apt-get update', shell=True, check=False)
    run_step('install ffmpeg', 'apt-get install -y ffmpeg', shell=True)
    run_step('show existing torch stack', [sys.executable, '-m', 'pip', 'show', 'torch', 'torchaudio', 'torchvision'], check=False)
    run_step('install compatible torch stack', [
        sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', '--progress-bar', 'on',
        'torch==2.8.0', 'torchaudio==2.8.0', 'torchvision==0.23.0',
    ])
    run_step('install WhisperX and video-descriptor', [
        sys.executable, '-m', 'pip', 'install', '--upgrade', '--upgrade-strategy', 'only-if-needed',
        '--progress-bar', 'on', '-r', REQ_URL, REPO_URL,
    ])
    run_step('show installed versions before restart', [sys.executable, '-m', 'pip', 'show', 'torch', 'torchaudio', 'whisperx', 'video-descriptor'], check=False)
    open(MARKER, 'w').close()
    print('Instalacion completada. Colab reiniciara el runtime ahora; vuelve a ejecutar esta misma celda despues del reinicio.')
    os.kill(os.getpid(), 9)

import numpy, pandas, torch, torchaudio, whisperx
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('whisperx', getattr(whisperx, '__version__', 'installed'))
assert hasattr(torchaudio, 'AudioMetaData'), 'torchaudio must be 2.8.x for pyannote.audio 3.3.2'
print('Entorno listo.')

## Subida interactiva

Esta ruta es cómoda, pero con vídeos largos puede fallar por navegador, red, sesión o disco temporal. Si falla, usa la celda de Drive/URL de abajo.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se ha subido ningún archivo")

VIDEO_PATH = next(iter(uploaded.keys()))
print(f"Video seleccionado: {VIDEO_PATH}")

## Fallback para vídeos grandes

Si la subida interactiva no aguanta el tamaño del vídeo, sube el archivo a Google Drive y copia una versión local a `/content` antes de ejecutar.

In [ ]:
# Opción Drive. Descomenta y ajusta la ruta si la subida interactiva falla.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/path/to/video.mp4" "/content/video.mp4"
# VIDEO_PATH = "/content/video.mp4"

# Opción URL descargable. Descomenta y ajusta si tienes una URL directa.
# !wget -O "/content/video.mp4" "https://example.com/video.mp4"
# VIDEO_PATH = "/content/video.mp4"

## Transcripción

Valores por defecto: español (`es`), `large-v3-turbo`, sin diarización.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('/content/video_descriptor_outputs')

!python -m video_descriptor transcribe "$VIDEO_PATH" \
  --output-dir "$OUTPUT_DIR" \
  --model large-v3-turbo \
  --language es \
  --batch-size 16 \
  --compute-type float16 \
  --device auto

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/video_descriptor_outputs', 'zip', OUTPUT_DIR)
files.download(archive)